# 02 — Feature Analysis

Build the wallet feature matrix and stress-test it before clustering.

**Inputs:** the four ingested parquets in `data/processed/` (transactions, token_transfers, receipts, address_codes).

**Outputs:** `wallet_features.parquet` and `wallet_features_scaled.parquet` — the latter feeds `03_clustering.ipynb`.

## Setup

In [13]:
# Tool Imports
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

# Project imports
# sys.path.insert(0, str(Path.cwd().parent)) # inserts github project parent to PATH
from src.features import build_features, scale_features
from src.models import fit_kmeans, cluster_profiles

# pandas display options: max_columns=50, max_colwidth=20
pd.options.display.max_columns = 50
pd.options.display.max_colwidth = 20
PROCESSED = Path("../data/processed")

## 1. Load features

Build the feature matrix if it doesn't exist yet, otherwise reload from disk.
This pattern keeps notebook re-runs fast.

In [ ]:
# If wallet_features.parquet exists -> pd.read_parquet
# Else -> build_features() and save
# Print shape and column list

features = None  # TODO

In [ ]:
# Quick sanity glance
# - features.head()
# - features.dtypes  (no unexpected object columns)
# - features.isna().sum()  (should be all zeros)

## 2. Raw feature distributions

Looking for: heavy tails (need log), zero-inflation (suspicious), bimodality (potential cluster signal).

In [ ]:
# Define the feature columns you'll inspect (mirror FEATURE_COLS from src/features/scaling.py)
FEATURE_COLS = [
    # TODO: paste from src/features/scaling.py
]

In [ ]:
# Histogram grid of raw features
# - 4x4 subplots (or similar), one feature per axis
# - bins=50
# - log y-scale if a tail dominates
#
# What to look for:
# - single spike at zero -> dead feature
# - long right tail -> log-transform needed
# - visible bimodality -> potential cluster axis

In [ ]:
# Summary statistics
# features[FEATURE_COLS].describe(percentiles=[.01, .25, .5, .75, .9, .99])
#
# What to look for:
# - 99th percentile >> median -> heavy tail
# - min and median both 0 -> mostly empty

In [ ]:
# Zero-inflation check
# For each feature: (features[col] == 0).mean()
# Print as a sorted Series, descending
#
# Decision: any feature with >95% zeros is probably dead weight.
# Note candidates to drop.

## 3. Scaling

Log1p the heavy-tailed columns, then StandardScaler the whole matrix.
See `src/features/scaling.py` for the column lists.

In [ ]:
# Run scaling
# scaled = scale_features(features)
# wallets = scaled["wallet"]
# X = scaled.drop(columns="wallet")

In [ ]:
# Confirm scaling worked
# X.describe().loc[["mean", "std"]].round(3)
#
# Means ~ 0, stds ~ 1. If not, stop and debug upstream.

In [ ]:
# Histogram grid of scaled features (same layout as raw histograms)
#
# What to look for:
# - bell-ish shapes -> good
# - spike + outliers -> essentially binary, weak feature
# - long tail even after log -> consider clipping at 99th percentile

## 4. Correlation analysis

K-Means has no correlation regularization. Two near-duplicate features double-weight that axis.
Goal: identify and drop redundant features.

In [ ]:
# Correlation heatmap
# - X.corr()
# - sns.heatmap with annot=True, fmt=".2f", cmap="RdBu_r", center=0
# - figsize ~ (12, 10) so numbers are readable
#
# What to look for: any pair with |corr| > 0.9.
# Likely culprits: tx_count <-> active_days,
#                  tokens_sent_count <-> unique_tokens_sent,
#                  unique_recipients <-> tx_count

In [ ]:
# Programmatic correlation flagging
# - Take the upper triangle of X.corr().abs()
# - Stack to long form, filter > 0.9, sort descending
# - Print offending pairs
#
# Decision: for each high-corr pair, pick which to keep (usually the more interpretable one).

### Feature pruning decisions

_Write 3-5 bullets here documenting what you're dropping and why. Example:_

- Dropping `active_days` (corr 0.94 with `tx_count`, less interpretable)
- Dropping `unique_tokens_sent` (corr 0.92 with `tokens_sent_count`)
- Keeping `failed_tx_ratio` despite low variance — strong bot signal
- ...

In [ ]:
# Apply pruning (if you decided to drop features above)
FINAL_FEATURES = [
    # TODO: final list after pruning
]

# X = X[FINAL_FEATURES]

## 5. Dimensionality preview (PCA)

30-second answer to "is there structure here?"
If 2-3 PCs explain >70% variance, K-Means will likely find clean clusters.
If you need 10+ PCs, expect noisier clusters.

In [ ]:
# PCA scree plot
# - PCA(n_components=len(FINAL_FEATURES)).fit(X)
# - Plot cumulative explained variance vs # components
# - Add horizontal lines at 0.7, 0.8, 0.9
#
# Read off: how many PCs to hit 80%?

In [ ]:
# PCA 2D scatter
# - Project X to 2 components
# - Scatter alpha=0.3, small markers
# - Color by one feature you expect to dominate (e.g., log(tx_count) or contract_call_ratio)
#
# What to look for: clumps, arms, density variations.
# Uniform blob = clustering will struggle.
# Fingers/islands = good signs.

## 6. Clusterability preview

Quick K-Means at k=6 to confirm features carry signal.
**Not the final clustering** — that's `03_clustering.ipynb`. This is a smoke test.

In [ ]:
# Quick K-Means at k=6
# from src.models import fit_kmeans
# model, labels = fit_kmeans(X, k=6)
#
# Print cluster sizes: pd.Series(labels).value_counts().sort_index()
#
# What to look for: sizes in 100s to 1000s, no single cluster with 95%+ of wallets.

In [ ]:
# Preview cluster profiles
# from src.models import cluster_profiles
# profiles = cluster_profiles(scaled, labels)  # pass scaled DF, including wallet column
#
# Display as heatmap: center=0, cmap="RdBu_r", annot=True, fmt=".1f"
#
# What to look for:
# - Each cluster should have 2-3 features clearly +/- vs global mean
# - If every cluster looks the same -> features don't discriminate, go back and improve
# - If you can already squint and see archetypes (high-activity, whale-like, etc.) -> ready for 03

## 7. Final feature set lock-in

_Fill in:_

- **Wallets in final matrix:** N
- **Features kept:** ...
- **Features dropped (and why):** ...
- **Clusterability assessment:** (one sentence based on sections 5-6)

In [ ]:
# Save final scaled matrix
# - Reattach wallet column to X (if you pruned, the columns are now FINAL_FEATURES)
# - Write to PROCESSED / "wallet_features_scaled.parquet"
# - This overwrites the version produced by scale_features() — intentional.

## Handoff to 03_clustering.ipynb

_Write 2-3 sentences for future-you:_

- Path to scaled features: `data/processed/wallet_features_scaled.parquet`
- Expected number of clusters based on PCA: ...
- First Etherscan-validation candidates: wallets with extreme `tx_count` and `failed_tx_ratio`